# Pronóstico horario de PM2.5 en Lima Metropolitana

El objetivo es estimar la concentración a **1, 3 y 6 horas** en las estaciones del SENAMHI. Cada modelo estima el cambio desde la última medición. El pronóstico es la concentración de emisión más ese cambio. Persistencia repite la lectura de emisión.

La selección empleó validación temporal en 2020, 2021 y 2022. Julio–septiembre de 2024 intervino en la comprobación de la selección. Octubre–diciembre de 2024 se reservó para la prueba independiente.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

import pandas as pd
from IPython.display import Image, display

PROYECTO = Path(os.environ.get("PM25_PROYECTO", Path.cwd())).resolve()
if Path("/content").is_dir() and not (PROYECTO / "codigo" / "evaluar_final.py").exists():
    PROYECTO = Path("/content/Viento-Lima")
    if not PROYECTO.exists():
        subprocess.run(["git", "clone", "--branch", "develop", "--single-branch",
                        "https://github.com/KeniChris/Viento-Lima.git", str(PROYECTO)], check=True)
    elif not (PROYECTO / "codigo" / "evaluar_final.py").exists():
        subprocess.run(["git", "-C", str(PROYECTO), "pull", "--ff-only", "origin", "develop"], check=True)
if not (PROYECTO / "codigo" / "evaluar_final.py").is_file():
    raise FileNotFoundError("Ejecuta el cuaderno desde Viento-Lima o configura PM25_PROYECTO.")

RESULTADOS = PROYECTO / "resultados"
PRONOSTICO = RESULTADOS / "pronostico"
FIGURAS = PRONOSTICO / "figuras"

## Datos y preparación

Se cruzaron las series por estación y hora en **America/Lima**. La hora del CSV original se verificó con OpenAQ antes de convertirla desde UTC. En cada ampliación se comprobó estación, unidad, hora entera, unicidad y valores no negativos. Las horas sin lectura se conservaron como faltantes y los picos válidos permanecieron en la serie.

La prueba de octubre a diciembre usa las mediciones de Campo de Marte, San Borja, San Juan de Lurigancho y San Martín de Porres. La matriz exige una lectura en la emisión y otra en la hora objetivo. La imputación y el escalado se ajustan en el entrenamiento de cada pliegue.

In [ ]:
display(pd.read_csv(RESULTADOS / "historico" / "verificacion_horaria.csv"))
display(pd.read_csv(PRONOSTICO / "eda_openaq_oct_dic_2024.csv").round(3))

## Modelado

**XGBoost.** Un conjunto de árboles añade correcciones sucesivas al cambio estimado. La figura presenta el primer árbol del modelo entrenado a una hora. Su primera división usa el cambio reciente de PM2.5. Cada hoja aporta una corrección al ensamble.

**LSTM.** La red procesa de la hora más antigua a la emisión. Cada paso contiene PM2.5 estandarizado y un indicador de disponibilidad. El estado de la última hora se concatena con rezagos, calendario y meteorología. Se ajustó una red por horizonte.

El contraste de pesos asignó un peso de 3 a los objetivos por encima del P95 de desarrollo por estación. La variante ponderada se adoptó cuando redujo el MAE macro y el MAE de episodios altos en validación.

In [ ]:
for filename in ("arbol_xgboost.png", "arquitectura_lstm.png"):
    display(Image(filename=str(FIGURAS / filename), width=850))

display(pd.DataFrame(json.loads((PRONOSTICO / "modelos.json").read_text(encoding="utf-8"))))
display(pd.read_csv(PRONOSTICO / "validacion_pesos.csv").round(3))

## Evaluación independiente

El MAE mide el error absoluto medio en µg/m³. El RMSE aumenta el peso de errores grandes. El residuo se calcula como pronosticado menos observado. Los residuos negativos en lecturas altas indican subestimación.

La serie compara un mismo episodio a distintos horizontes. El caso se fijó por el máximo observado de la prueba a una hora. En la dispersión, la diagonal marca coincidencia con la medición. Los residuos de episodios altos aparecen resaltados.

In [ ]:
display(pd.read_csv(PRONOSTICO / "prueba.csv").round(3))
display(pd.read_csv(PRONOSTICO / "diferencia_persistencia.csv").round(3))
por_estacion = pd.read_csv(PRONOSTICO / "por_estacion.csv")
display(por_estacion.loc[por_estacion.modelo.ne("persistencia")].round(3))

for filename in ("episodio_observado.png", "ajuste_y_residuos.png"):
    display(Image(filename=str(FIGURAS / filename), width=850))

## Interrupciones de medición

En ventanas de 24 horas completas se simularon cortes consecutivos de 1, 6 y 12 horas antes de emitir. Se mantuvieron la lectura de emisión, las horas objetivo, el clima y las lecturas vecinas. Se recalcularon los rezagos y resúmenes afectados. Cada fila del contraste usa los mismos casos que su condición sin corte.

In [ ]:
faltantes = pd.read_csv(PRONOSTICO / "faltantes.csv")
display(faltantes[["horizonte_h", "corte_h", "n", "MAE", "delta_MAE", "IC95_inferior", "IC95_superior"]].round(3))

## Alertas de propagación

Tras un alza anómala en origen se examina si otra estación presenta un alza en las seis horas siguientes. Se comparan la regla física y dos clasificadores logísticos, con y sin rumbo del viento. El umbral se fijó en 2022. La matriz resume los casos con toda la ventana objetivo observada.

In [ ]:
matriz = pd.read_csv(RESULTADOS / "alertas" / "matriz_confusion.csv")
display(matriz.loc[matriz.periodo.eq("prueba_independiente"),
    ["metodo", "aciertos", "falsas_alertas", "falsos_negativos", "verdaderos_negativos",
     "precision", "sensibilidad"]].round(3))

## Integración con Redis

Redis guarda observaciones horarias, métricas y pronósticos. La consulta vectorial compara PM2.5 reciente, humedad y componentes del viento. Devuelve episodios históricos y su concentración medida seis horas después.

La verificación reconstruye las entradas desde Redis, carga los modelos ajustados y compara sus tres salidas con los pronósticos de la evaluación independiente. El repositorio incluye los comandos de carga, consulta y verificación.

In [ ]:
redis_verificacion = json.loads((PRONOSTICO / "redis.json").read_text(encoding="utf-8"))
display(pd.DataFrame(redis_verificacion["pronosticos"]).round(5))
display(pd.Series({key: redis_verificacion[key] for key in (
    "observaciones_cargadas", "episodios_indexados", "vecinos_recuperados",
    "consulta_coincide_con_busqueda_exacta", "repeticion_sin_duplicados", "diferencia_maxima_ug_m3",
)}))